In [18]:
> CV generati con IA

SyntaxError: invalid syntax (2205268368.py, line 1)

In [6]:
import os

os.makedirs("resumes", exist_ok=True)

cvs = {
    "curriculum-marketing.txt": """### MARIO ROSSI
Digital Marketing Manager
Email: mario.rossi@email.com
Tel: +39 333 1234567
Milano, Italia
### PROFILO PROFESSIONALE
Senior Digital Marketing Manager con 15 anni di esperienza. Specializzato in SEO, SEM, social media marketing e content marketing. Guida team cross-funzionali e lancia prodotti con campagne digitali innovative.
### COMPETENZE
SEO, Google Ads, Meta Ads, email marketing, Google Analytics, brand positioning, gestione budget pubblicitari.
### ESPERIENZE
Responsabile marketing in un'azienda e-commerce: strategia di lancio prodotti, crescita del traffico organico del 120%.
""",
    "curriculum-cybersecurity.txt": """### ANDREA BIANCHI
Information Security Manager
Email: andrea.bianchi@email.com
Tel: +39 345 9876543
Roma, Italia
### PROFILO PROFESSIONALE
Esperto di sicurezza informatica con 10 anni di esperienza. Certificato CEH e CISSP. Si occupa di protezione di infrastrutture, siti web e applicazioni aziendali.
### COMPETENZE
Penetration testing, vulnerability assessment, hardening di server, firewall, gestione incidenti, conformità GDPR e ISO 27001.
### ESPERIENZE
Responsabile sicurezza in una banca: audit periodici, messa in sicurezza di portali web e risposta ad attacchi.
""",
    "curriculum-software-engineer.txt": """### LUCA FERRARI
Senior Software Engineer - Web Applications
Email: luca.ferrari@email.com
Tel: +39 347 8565432
Torino, Italia
### PROFILO PROFESSIONALE
Sviluppatore web con 8 anni di esperienza nella progettazione di applicazioni scalabili, sia backend che frontend.
### COMPETENZE
Python, PHP, Laravel, JavaScript, React, SQL, Docker, API REST, test automatizzati.
### ESPERIENZE
Sviluppo di piattaforme SaaS e migrazione di sistemi legacy verso architetture a microservizi.
""",
}

for name, text in cvs.items():
    with open(os.path.join("resumes", name), "w", encoding="utf-8") as f:
        f.write(text)

print(os.listdir("resumes"))

['curriculum-cybersecurity.txt', 'curriculum-marketing.txt', 'curriculum-software-engineer.txt']


In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

load_dotenv()
openai_key = os.getenv("OPENAI_API_KEY")
print(openai_key is not None)

True


In [8]:
documents_dir = "resumes"
documents, metadatas, ids = [], [], []
id = 0

for filename in os.listdir(documents_dir):
    if filename.endswith(".txt"):
        with open(os.path.join(documents_dir, filename), "r", encoding="utf-8") as file:
            chunks = file.read().replace("\n", ".").split("### ")
            for chunk in chunks:
                if chunk and not chunk.isspace():
                    documents.append(chunk)
                    metadatas.append({"source": filename, "info": chunks[1]})
                    ids.append(str(id))
                    id += 1

print(len(documents), len(metadatas), len(ids))

12 12 12


In [9]:
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="CVs",
    embedding_function=openai_ef
)

In [10]:
collection.add(documents=documents, metadatas=metadatas, ids=ids)

In [11]:
collection.count()


12

In [13]:
user_question = "mi serve qualcuno per mettere in sicurezza il mio sito web"

results = collection.query(
    query_texts=[user_question],
    n_results=1
)
results

{'ids': [['1']],
 'embeddings': None,
 'documents': [['PROFILO PROFESSIONALE.Esperto di sicurezza informatica con 10 anni di esperienza. Certificato CEH e CISSP. Si occupa di protezione di infrastrutture, siti web e applicazioni aziendali..']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'curriculum-cybersecurity.txt',
    'info': 'ANDREA BIANCHI.Information Security Manager.Email: andrea.bianchi@email.com.Tel: +39 345 9876543.Roma, Italia.'}]],
 'distances': [[0.5217217206954956]]}

In [16]:
best = results["metadatas"][0][0]

context = (
    f"CONTESTO: nome file {best['source']}. "
    f"Paragrafo più significativo: {results['documents'][0][0]}. "
    f"Menziona il nome del candidato all'inizio e i dati personali alla fine per il contatto: {best['info']}"
)

prompt = f"""Dato il seguente contesto {context} rispondi alla domanda dell'utente {user_question} 
spiegando che nel file individuato c'è il profilo più adatto.
Argomenta la scelta usando il contenuto del testo individuato nel contesto."""
prompt

"Dato il seguente contesto CONTESTO: nome file curriculum-cybersecurity.txt. Paragrafo più significativo: PROFILO PROFESSIONALE.Esperto di sicurezza informatica con 10 anni di esperienza. Certificato CEH e CISSP. Si occupa di protezione di infrastrutture, siti web e applicazioni aziendali... Menziona il nome del candidato all'inizio e i dati personali alla fine per il contatto: ANDREA BIANCHI.Information Security Manager.Email: andrea.bianchi@email.com.Tel: +39 345 9876543.Roma, Italia. rispondi alla domanda dell'utente mi serve qualcuno per mettere in sicurezza il mio sito web \nspiegando che nel file individuato c'è il profilo più adatto.\nArgomenta la scelta usando il contenuto del testo individuato nel contesto."

In [17]:
client = OpenAI(api_key=openai_key)

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": "Sei un assistente HR, specializzato nella ricerca di profili professionali"},
        {"role": "user", "content": prompt},
    ],
)

print(completion.choices[0].message.content)

Ciao! Ho trovato il profilo di un candidato che potrebbe essere perfetto per aiutarti a mettere in sicurezza il tuo sito web. Si tratta di **Andrea Bianchi**, un esperto di sicurezza informatica con ben 10 anni di esperienza nel settore. 

Nel suo profilo professionale, Andrea è descritto come un professionista certificato CEH (Certified Ethical Hacker) e CISSP (Certified Information Systems Security Professional), il che dimostra una solida preparazione nelle best practices di sicurezza informatica. Queste certificazioni sono molto rispettate e indicano che possiede competenze avanzate nella protezione di infrastrutture critiche, siti web e applicazioni aziendali, proprio ciò di cui hai bisogno.

Inoltre, la sua esperienza specifica nella protezione di siti web suggerisce che sia ben equipaggiato per affrontare le sfide di sicurezza che potrebbero presentarsi sul tuo sito. Se desideri entrare in contatto con lui, puoi trovarlo a Roma, Italia, e raggiungerlo via email all'indirizzo and